# 07 数值微分实验汇总

本 Notebook 汇总第五章的主要实验：差分公式收敛阶、步长 U 形曲线、端点与内部节点比较、Richardson 外推、离散数据噪声、样条微分和隐式紧致差分。


In [ ]:
import math
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src" / "py_sc").exists():
        ROOT = candidate
        break
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import (
    central_difference,
    compact_first_derivative_periodic,
    differentiate_discrete,
    five_point_center_derivative,
    forward_difference,
    natural_cubic_spline_derivative,
    observed_order,
    richardson_derivative,
    second_derivative_five_point,
    three_point_endpoint_derivative,
)


## 实验 1：前向、中心和五点公式收敛阶

使用 $f(x)=\sin x$，在固定点 $x=0.7$ 处比较三种一阶公式。理论上前向差分是一阶，中心差分是二阶，五点中心公式是四阶。


In [ ]:
x0 = 0.7
exact = math.cos(x0)
h_values = 2.0 ** (-np.arange(3, 11, dtype=float))
forward_errors = np.array([abs(float(forward_difference(np.sin, x0, h)) - exact) for h in h_values])
central_errors = np.array([abs(float(central_difference(np.sin, x0, h)) - exact) for h in h_values])
five_errors = np.array([abs(float(five_point_center_derivative(np.sin, x0, h)) - exact) for h in h_values])

print("forward orders:", observed_order(forward_errors))
print("central orders:", observed_order(central_errors))
print("five-point orders:", observed_order(five_errors))

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(h_values, forward_errors, "o-", label="前向差分")
ax.loglog(h_values, central_errors, "s-", label="中心差分")
ax.loglog(h_values, five_errors, "^-", label="五点中心")
ax.invert_xaxis()
ax.set_xlabel("步长 h")
ax.set_ylabel("绝对误差")
ax.set_title("不同一阶公式的收敛阶")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
plt.show()


## 实验 2：步长过小时舍入误差主导

中心差分的截断误差是 $O(h^2)$，但舍入误差会近似按 $O(\epsilon/h)$ 放大。下面用 $e^x$ 展示误差随步长变化的 U 形曲线。


In [ ]:
x0 = 1.0
exact = math.exp(x0)
h_roundoff = 10.0 ** (-np.arange(1, 17, dtype=float))
roundoff_errors = np.array([abs(float(central_difference(np.exp, x0, h)) - exact) for h in h_roundoff])

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(h_roundoff, roundoff_errors, marker="o")
ax.invert_xaxis()
ax.set_xlabel("步长 h")
ax.set_ylabel("绝对误差")
ax.set_title("步长过小时舍入误差开始主导")
ax.grid(True, which="both", alpha=0.3)
plt.show()


## 实验 3：端点公式和内部公式

同一个数据区间内，内部点可以使用中心公式，端点需要单边公式。端点公式通常需要更多邻近点来达到较高精度。


In [ ]:
h = 0.05
left = float(three_point_endpoint_derivative(np.sin, 0.0, h, side="left"))
center = float(central_difference(np.sin, math.pi / 4, h))
right = float(three_point_endpoint_derivative(np.sin, math.pi, h, side="right"))
print("左端点误差：", abs(left - 1.0))
print("内部点误差：", abs(center - math.cos(math.pi / 4)))
print("右端点误差：", abs(right + 1.0))


## 实验 4：Richardson 外推

中心差分有规则的偶次误差展开，因此 Richardson 外推可以明显提高精度。外推表的第一列是原始中心差分，后续列逐步消去 $h^2,h^4,\ldots$ 误差项。


In [ ]:
result = richardson_derivative(np.sin, 0.7, h=0.25, levels=5)
print(result.table)
print("外推误差：", abs(result.value - math.cos(0.7)))


## 实验 5：离散数据噪声

这里比较无噪声和含噪声数据的五点离散微分。即使原始噪声幅度很小，导数估计也会出现更明显的振荡。


In [ ]:
rng = np.random.default_rng(2026)
x = np.linspace(0.0, 2.0 * math.pi, 81)
clean = np.sin(x)
noisy = clean + 0.02 * rng.normal(size=x.size)
exact_d1 = np.cos(x)
clean_d1 = differentiate_discrete(x, clean, stencil_size=5)
noisy_d1 = differentiate_discrete(x, noisy, stencil_size=5)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, exact_d1, label="解析导数")
ax.plot(x, clean_d1, "--", label="无噪声差分")
ax.plot(x, noisy_d1, ":", label="含噪声差分")
ax.set_title("离散数据微分中的噪声放大")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


## 实验 6：直接有限差分与样条微分

样条微分先构造全局 $C^2$ 的分段三次函数，再对每段解析求导。它常比逐点差分更平滑，但插值样条仍会穿过噪声点。


In [ ]:
x_nodes = np.linspace(0.0, 2.0 * math.pi, 25)
y_nodes = np.sin(x_nodes)
x_eval = np.linspace(0.0, 2.0 * math.pi, 300)
spline_d1 = natural_cubic_spline_derivative(x_nodes, y_nodes, x_eval)
finite_d1 = differentiate_discrete(x_nodes, y_nodes, stencil_size=5)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x_eval, np.cos(x_eval), label="解析导数")
ax.plot(x_eval, spline_d1, "--", label="样条导数")
ax.scatter(x_nodes, finite_d1, s=25, label="节点差分")
ax.set_title("样条微分与直接差分")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


## 实验 7：二阶公式和紧致差分

二阶五点公式验证二阶导数近似；周期紧致差分展示隐式格式在一阶导数上的高精度。两者都会在后续微分方程离散化中再次出现。


In [ ]:
second_error = abs(float(second_derivative_five_point(np.sin, 0.7, 0.05)) + math.sin(0.7))
print("二阶五点误差：", second_error)

n = 64
x_periodic = np.linspace(0.0, 2.0 * math.pi, n, endpoint=False)
h_periodic = x_periodic[1] - x_periodic[0]
y_periodic = np.sin(x_periodic)
compact = compact_first_derivative_periodic(y_periodic, h_periodic)
print("周期紧致差分最大误差：", np.max(np.abs(compact - np.cos(x_periodic))))


## 本章实验小结

这些实验说明：

* 公式阶数可以通过步长减半实验验证；
* 步长过小时，舍入误差会破坏收敛；
* 端点和内部节点需要不同模板；
* Richardson 外推能提高光滑函数上的差分精度；
* 噪声会被差分放大，二阶导数更敏感；
* 样条微分适合平滑重构，但不是去噪方法；
* 隐式紧致差分可以用较短模板获得高精度，但需要处理线性系统和边界条件。
